# Input Resolution Study
This notebook performs experiments to find out the influence that the size of input resolution brings to model.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.model_selection import train_test_split
from google.colab import drive



drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/CA Notebook/image"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")


Mounted at /content/drive


random seed

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True # Ensure deterministic convolution algorithms
    torch.backends.cudnn.benchmark = False

seed_everything(42) # Lock random seed for reproducibility

In [ ]:
from torch.utils.data import Dataset
class FruitDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        '''
        Custom dataset for fruit classification.
        Image label is inferred from filename prefix.
        '''
        self.image_files = [f for f in os.listdir(root_dir) if f.lower().endswith(".jpg")]
        self.class_map = {"apple": 0, "banana": 1, "orange": 2, "mixed": 3}
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        filename = self.image_files[idx]
        image_path = os.path.join(self.root_dir, filename)
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Label parsing logic
        label = -1
        for cls, idx_val in self.class_map.items():
            if filename.lower().startswith(cls):
                label = idx_val
                break
        return image, label

def get_loaders(root_dir, img_size, batch_size):
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])

    # Load the full dataset first for split
    full_dataset = FruitDataset(root_dir, transform=transform)
    indices = list(range(len(full_dataset)))

    # Extract labels for Stratified Split
    labels = []
    for f in full_dataset.image_files:
        for cls, idx in full_dataset.class_map.items():
            if f.lower().startswith(cls):
                labels.append(idx)
                break

    # Split into 80% training and 20% validation set
    train_idx, val_idx = train_test_split(
        indices, test_size=0.2, stratify=labels, random_state=42
    )

    train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

In [ ]:
import copy
import matplotlib.pyplot as plt

# Global storage dictionary
if 'res_results' not in locals():
    res_results = {}

print("Resolutin experiment memory is ready.")

Resolutin experiment memory is ready.


In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(dataloader), correct / total

def evaluate(model, dataloader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [ ]:
class DeepFruitCNN(nn.Module):
    def __init__(self, num_layers=2, num_classes=4):
        super(DeepFruitCNN, self).__init__()

        layers = []
        in_channels = 3
        # Filter sequence for selecting different network depths
        filters = [16, 32, 64, 128]

        for i in range(num_layers):
            layers.append(nn.Conv2d(in_channels, filters[i], kernel_size=3, padding=1))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2, 2))
            in_channels = filters[i]

        self.conv_section = nn.Sequential(*layers)

        # AdaptiveAvgPool2d ensures fixed input for the fully connected layer, independent of the number of convolutional layers
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.fc = nn.Sequential(
            nn.Linear(filters[num_layers-1] * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_section(x)
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

32*32

In [ ]:
# ================= Manual Modification Area =================
CURRENT_SIZE = 32      # <--- Manually modify to 32, 64, 96, or 128
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.001
# ===============================================
# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42) # Lock random seed to ensure fair comparison

# 1. Get DataLoaders for the current image size
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, BATCH_SIZE)

# 2. Get Test Loader for the current image size (Resize must be synchronized)
test_transform = transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor()
])
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=test_transform),
                         batch_size=BATCH_SIZE, shuffle=False)

# 3. Initialize 3-layer model
model = DeepFruitCNN(num_layers=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 4. Training variables
best_val_acc = 0.0
best_model_wts = None
best_epoch = 0
history = {'val_acc': []}

print(f"\n🚀 Start testing resolution: {CURRENT_SIZE}x{CURRENT_SIZE}...")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    v_acc = evaluate(model, val_loader)
    history['val_acc'].append(v_acc)

    # Best model checkpoint logic
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        # Copy the current best model weights
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] Loss: {t_loss:.4f} | Val Acc: {v_acc:.4f} ")

print(f"Best val acc: {best_val_acc:.4f}")
# 5. Load the best weights for this size to evaluate on the test set
model.load_state_dict(best_model_wts)
final_test_acc = evaluate(model, test_loader)

# 6. Save results to the global storage dictionary
res_results[CURRENT_SIZE] = {
    'val_history': history['val_acc'],
    'best_val': best_val_acc,
    'best_epoch': best_epoch,
    'test_acc': final_test_acc
}

print(f"\n✅ {CURRENT_SIZE}x{CURRENT_SIZE} experiment finished！")
print(f"The best performance of this size appears in the {best_epoch}th epoch，Test Acc: {final_test_acc:.4f}")


🚀 Start testing resolution: 32x32...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Loss: 1.3461 | Val Acc: 0.6042 
Epoch [02/30] Loss: 1.2537 | Val Acc: 0.3125 
Epoch [03/30] Loss: 1.0166 | Val Acc: 0.7500 
Epoch [04/30] Loss: 0.7578 | Val Acc: 0.7083 
Epoch [05/30] Loss: 0.5532 | Val Acc: 0.8125 
Epoch [06/30] Loss: 0.4025 | Val Acc: 0.8542 
Epoch [07/30] Loss: 0.4285 | Val Acc: 0.8542 
Epoch [08/30] Loss: 0.2916 | Val Acc: 0.8542 
Epoch [09/30] Loss: 0.2572 | Val Acc: 0.8958 
Epoch [10/30] Loss: 0.2527 | Val Acc: 0.8542 
Epoch [11/30] Loss: 0.2078 | Val Acc: 0.8958 
Epoch [12/30] Loss: 0.1632 | Val Acc: 0.8958 
Epoch [13/30] Loss: 0.1466 | Val Acc: 0.8750 
Epoch [14/30] Loss: 0.1334 | Val Acc: 0.8958 
Epoch [15/30] Loss: 0.1020 | Val Acc: 0.8958 
Epoch [16/30] Loss: 0.1273 | Val Acc: 0.9167 
Epoch [17/30] Loss: 0.1003 | Val Acc: 0.8750 
Epoch [18/30] Loss: 0.1073 | Val Acc: 0.8750 
Epoch [19/30] Loss: 0.0916 | Val Acc: 0.8958 
Epoch [20/30] Loss: 0.0818 | Val Acc: 0.8542 
Epoch [21/30] Loss: 0.0784 | Val Acc: 0.8958 
Epoch [22/30] Loss: 0.0528 | Val A

64*64

In [ ]:
# ================= Manual Modification Area =================
CURRENT_SIZE = 64     # <--- Manually modify to 32, 64, 96, or 128
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.001
# ===============================================
# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42) # Lock random seed to ensure fair comparison

# 1. Get DataLoaders for the current image size
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, BATCH_SIZE)

# 2. Get Test Loader for the current image size (Resize must be synchronized)
test_transform = transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor()
])
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=test_transform),
                         batch_size=BATCH_SIZE, shuffle=False)

# 3. Initialize 3-layer model
model = DeepFruitCNN(num_layers=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 4. Training variables
best_val_acc = 0.0
best_model_wts = None
best_epoch = 0
history = {'val_acc': []}

print(f"\n🚀 Start testing resolution: {CURRENT_SIZE}x{CURRENT_SIZE}...")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    v_acc = evaluate(model, val_loader)
    history['val_acc'].append(v_acc)

    # Best model checkpoint logic
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        # Copy the current best model weights
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] Loss: {t_loss:.4f} | Val Acc: {v_acc:.4f} ")

print(f"Best val acc: {best_val_acc:.4f}")
# 5. Load the best weights for this size to evaluate on the test set
model.load_state_dict(best_model_wts)
final_test_acc = evaluate(model, test_loader)

# 6. Save results to the global storage dictionary
res_results[CURRENT_SIZE] = {
    'val_history': history['val_acc'],
    'best_val': best_val_acc,
    'best_epoch': best_epoch,
    'test_acc': final_test_acc
}

print(f"\n✅ {CURRENT_SIZE}x{CURRENT_SIZE} experiment finished！")
print(f"The best performance of this size appears in the {best_epoch}th epoch，Test Acc: {final_test_acc:.4f}")


🚀 Start testing resolution: 64x64...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Loss: 1.3451 | Val Acc: 0.5833 
Epoch [02/30] Loss: 1.2483 | Val Acc: 0.3125 
Epoch [03/30] Loss: 1.0023 | Val Acc: 0.7917 
Epoch [04/30] Loss: 0.7449 | Val Acc: 0.7500 
Epoch [05/30] Loss: 0.5213 | Val Acc: 0.8333 
Epoch [06/30] Loss: 0.3705 | Val Acc: 0.8333 
Epoch [07/30] Loss: 0.4129 | Val Acc: 0.8333 
Epoch [08/30] Loss: 0.2662 | Val Acc: 0.8750 
Epoch [09/30] Loss: 0.2124 | Val Acc: 0.9167 
Epoch [10/30] Loss: 0.1724 | Val Acc: 0.8750 
Epoch [11/30] Loss: 0.1435 | Val Acc: 0.9167 
Epoch [12/30] Loss: 0.1169 | Val Acc: 0.8958 
Epoch [13/30] Loss: 0.1094 | Val Acc: 0.9167 
Epoch [14/30] Loss: 0.0861 | Val Acc: 0.9167 
Epoch [15/30] Loss: 0.0750 | Val Acc: 0.9167 
Epoch [16/30] Loss: 0.1000 | Val Acc: 0.9167 
Epoch [17/30] Loss: 0.0994 | Val Acc: 0.8750 
Epoch [18/30] Loss: 0.1065 | Val Acc: 0.8958 
Epoch [19/30] Loss: 0.0808 | Val Acc: 0.9167 
Epoch [20/30] Loss: 0.0657 | Val Acc: 0.9583 
Epoch [21/30] Loss: 0.0477 | Val Acc: 0.9375 
Epoch [22/30] Loss: 0.0376 | Val A

96*96

In [ ]:
# ================= Manual Modification Area =================
CURRENT_SIZE = 96      # <--- Manually modify to 32, 64, 96, or 128
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.001
# ===============================================
# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42) # Lock random seed to ensure fair comparison

# 1. Get DataLoaders for the current image size
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, BATCH_SIZE)

# 2. Get Test Loader for the current image size (Resize must be synchronized)
test_transform = transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor()
])
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=test_transform),
                         batch_size=BATCH_SIZE, shuffle=False)

# 3. Initialize 3-layer model
model = DeepFruitCNN(num_layers=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 4. Training variables
best_val_acc = 0.0
best_model_wts = None
best_epoch = 0
history = {'val_acc': []}

print(f"\n🚀 Start testing resolution: {CURRENT_SIZE}x{CURRENT_SIZE}...")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    v_acc = evaluate(model, val_loader)
    history['val_acc'].append(v_acc)

    # Best model checkpoint logic
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        # Copy the current best model weights
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] Loss: {t_loss:.4f} | Val Acc: {v_acc:.4f} ")

print(f"Best val acc: {best_val_acc:.4f}")
# 5. Load the best weights for this size to evaluate on the test set
model.load_state_dict(best_model_wts)
final_test_acc = evaluate(model, test_loader)

# 6. Save results to the global storage dictionary
res_results[CURRENT_SIZE] = {
    'val_history': history['val_acc'],
    'best_val': best_val_acc,
    'best_epoch': best_epoch,
    'test_acc': final_test_acc
}

print(f"\n✅ {CURRENT_SIZE}x{CURRENT_SIZE} experiment finished！")
print(f"The best performance of this size appears in the {best_epoch}th epoch，Test Acc: {final_test_acc:.4f}")


🚀 Start testing resolution: 96x96...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Loss: 1.3456 | Val Acc: 0.5833 
Epoch [02/30] Loss: 1.2449 | Val Acc: 0.3750 
Epoch [03/30] Loss: 0.9864 | Val Acc: 0.7917 
Epoch [04/30] Loss: 0.7277 | Val Acc: 0.7500 
Epoch [05/30] Loss: 0.5075 | Val Acc: 0.8333 
Epoch [06/30] Loss: 0.3559 | Val Acc: 0.8333 
Epoch [07/30] Loss: 0.4114 | Val Acc: 0.8750 
Epoch [08/30] Loss: 0.2743 | Val Acc: 0.8958 
Epoch [09/30] Loss: 0.2156 | Val Acc: 0.8958 
Epoch [10/30] Loss: 0.1574 | Val Acc: 0.8958 
Epoch [11/30] Loss: 0.1424 | Val Acc: 0.9167 
Epoch [12/30] Loss: 0.1062 | Val Acc: 0.9167 
Epoch [13/30] Loss: 0.0992 | Val Acc: 0.9167 
Epoch [14/30] Loss: 0.0748 | Val Acc: 0.9375 
Epoch [15/30] Loss: 0.0690 | Val Acc: 0.9167 
Epoch [16/30] Loss: 0.0948 | Val Acc: 0.9375 
Epoch [17/30] Loss: 0.0811 | Val Acc: 0.8750 
Epoch [18/30] Loss: 0.0780 | Val Acc: 0.9375 
Epoch [19/30] Loss: 0.0541 | Val Acc: 0.9167 
Epoch [20/30] Loss: 0.0567 | Val Acc: 0.9167 
Epoch [21/30] Loss: 0.0350 | Val Acc: 0.9583 
Epoch [22/30] Loss: 0.0277 | Val A

128*128

In [ ]:
# ================= Manual Modification Area =================
CURRENT_SIZE = 128      # <--- Manually modify to 32, 64, 96, or 128
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.001
# ===============================================
# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42) # Lock random seed to ensure fair comparison

# 1. Get DataLoaders for the current image size
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, BATCH_SIZE)

# 2. Get Test Loader for the current image size (Resize must be synchronized)
test_transform = transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor()
])
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=test_transform),
                         batch_size=BATCH_SIZE, shuffle=False)

# 3. Initialize 3-layer model
model = DeepFruitCNN(num_layers=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 4. Training variables
best_val_acc = 0.0
best_model_wts = None
best_epoch = 0
history = {'val_acc': []}

print(f"\n🚀 Start testing resolution: {CURRENT_SIZE}x{CURRENT_SIZE}...")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    v_acc = evaluate(model, val_loader)
    history['val_acc'].append(v_acc)

    # Best model checkpoint logic
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        # Copy the current best model weights
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] Loss: {t_loss:.4f} | Val Acc: {v_acc:.4f} ")

print(f"Best val acc: {best_val_acc:.4f}")
# 5. Load the best weights for this size to evaluate on the test set
model.load_state_dict(best_model_wts)
final_test_acc = evaluate(model, test_loader)

# 6. Save results to the global storage dictionary
res_results[CURRENT_SIZE] = {
    'val_history': history['val_acc'],
    'best_val': best_val_acc,
    'best_epoch': best_epoch,
    'test_acc': final_test_acc
}

print(f"\n✅ {CURRENT_SIZE}x{CURRENT_SIZE} experiment finished！")
print(f"The best performance of this size appears in the {best_epoch}th epoch，Test Acc: {final_test_acc:.4f}")


🚀 Start testing resolution: 128x128...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Loss: 1.3437 | Val Acc: 0.6667 
Epoch [02/30] Loss: 1.2245 | Val Acc: 0.3125 
Epoch [03/30] Loss: 0.9725 | Val Acc: 0.7500 
Epoch [04/30] Loss: 0.6911 | Val Acc: 0.7500 
Epoch [05/30] Loss: 0.4874 | Val Acc: 0.8333 
Epoch [06/30] Loss: 0.3431 | Val Acc: 0.8333 
Epoch [07/30] Loss: 0.4184 | Val Acc: 0.8750 
Epoch [08/30] Loss: 0.2769 | Val Acc: 0.8958 
Epoch [09/30] Loss: 0.2178 | Val Acc: 0.9375 
Epoch [10/30] Loss: 0.1472 | Val Acc: 0.9167 
Epoch [11/30] Loss: 0.1397 | Val Acc: 0.9375 
Epoch [12/30] Loss: 0.1039 | Val Acc: 0.9375 
Epoch [13/30] Loss: 0.1035 | Val Acc: 0.9167 
Epoch [14/30] Loss: 0.0734 | Val Acc: 0.9375 
Epoch [15/30] Loss: 0.0638 | Val Acc: 0.9167 
Epoch [16/30] Loss: 0.0866 | Val Acc: 0.9375 
Epoch [17/30] Loss: 0.0754 | Val Acc: 0.8750 
Epoch [18/30] Loss: 0.0772 | Val Acc: 0.9375 
Epoch [19/30] Loss: 0.0539 | Val Acc: 0.9375 
Epoch [20/30] Loss: 0.0640 | Val Acc: 0.8958 
Epoch [21/30] Loss: 0.0492 | Val Acc: 0.9583 
Epoch [22/30] Loss: 0.0351 | Val A

plot

In [ ]:
def plot_confusion_matrix(model, loader, device, classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, lbs in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbs.cpu().numpy())

    # Calculate confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)

    # Plot the confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix: Experiment 5 (Balanced Data & Early Stop)')
    plt.ylabel('True Label (Actual)')
    plt.xlabel('Predicted Label')
    plt.show()

# The class order must be consistent with your Dataset
class_names = ['apple', 'banana', 'orange', 'mixed']

# Run the plotting function directly

Experiments showed that although high resolution (96x96, 128x128) achieved a higher validation set peak (0.96), the lack of regularization led to severe overfitting and instability in the later stages of training, resulting in a test accuracy (0.90) slightly lower than that of the low resolution 32x32 (0.917). To preserve more image details to support the classification of subsequent complex categories, we chose 128x128 and 32*32 as the experimental size for future experiments and will introduce regularization in the next stage to suppress overfitting.